# 原始季度营收指引：来源与时间资格

唯一信息是每次季度财报原始下一财季营收指引的增长中点。它是按公司当时口径发布的数字，保留并购包含/排除假设，不重构可比范围；不使用后续公司实际营收、不使用分析师共识、不声称 latest current guide。

每目标财季只取第一次同时有已报告季度收入及 next-quarter guide 的披露。May6提前收入没有下一季指引，May8补发才生成该季指导事件。中期更新留作审计，不覆盖原始指导量。

采用披露/备案日期及原API/index日期较晚者后的首个实际交易日09:31NY；不修正有冲突的秒级时钟，也不回填同日16:15。所有非交易日及未知保留。

In [1]:
from pathlib import Path
import os,sys,json,hashlib
import pandas as pd,numpy as np,yaml
os.environ.setdefault('MLFLOW_ALLOW_FILE_STORE','true')
ROOT=Path.cwd()
if not (ROOT/'src').is_dir(): ROOT=ROOT.parents[1]
sys.path.insert(0,str(ROOT))
CARD=ROOT/'research/nvda_original_guidance_qualification_20261011'
plan=json.loads((CARD/'plan.json').read_text())
import qlib
from qlib.data import D
from qlib.workflow import R
from src.core.recorder_sources import load_verified_recorder_objects
from src.data.provider_content_hash import provider_content_hash
from src.data.issuer_guidance import read_guidance_facts,original_guidance_events,guidance_information_state,build_guidance_store,period_ordinal
from src.data.oos_horizon_inputs import prepare_horizon_inputs
assert provider_content_hash(ROOT/plan['price_provider'])[0]==plan['price_provider_content_sha256']
qlib.init(provider_uri=str(ROOT/plan['price_provider']),region='us',kernels=1)
metadata,metadata_audit=load_verified_recorder_objects(ROOT/plan['metadata_card'],expected_recorder_id=plan['metadata_recorder'],artifacts=['intake/earnings_item_inventory.pkl'])
manifest_path=ROOT/plan['fact_manifest'];manifest=json.loads(manifest_path.read_text())
raw_facts,normalized=read_guidance_facts(ROOT,manifest,as_of=plan['as_of'])
assert set(normalized.accession)==set(metadata['intake/earnings_item_inventory.pkl'].accessionNumber)
expected=[f'FY{i//4}Q{i%4+1}' for i in range(period_ordinal(plan['quarter_start']),period_ordinal(plan['quarter_end'])+1)]
events=original_guidance_events(normalized,expected_target_quarters=expected)
assert len(events)==63 and len(normalized)==66
bundle=prepare_horizon_inputs(ROOT,json.loads((ROOT/plan['parent_plan']).read_text()))
calendar=pd.DatetimeIndex(bundle['frame'].index)
state,available_events=guidance_information_state(events,calendar)
assert len(state)==3956 and state.guidance_qoq.loc[state.known.eq(0)].isna().all()
assert available_events.loc[available_events.accession.eq('0001045810-14-000076'),'available_at'].iloc[0]==pd.Timestamp('2014-05-09 13:31+00:00')
print('66 event facts,63 original next-quarter guides; known',int(state.known.sum()),'unknown',int(state.known.eq(0).sum()))


66 event facts,63 original next-quarter guides; known 3924 unknown 32


In [2]:
source=dict(fact_manifest=plan['fact_manifest'],fact_manifest_sha256=hashlib.sha256(manifest_path.read_bytes()).hexdigest(),source='SEC as-filed textual extracts and dated NVIDIA original release textual extracts',semantics='original management-defined guide growth, not latest current guide or consolidation-adjusted growth')
if (CARD/'summary.json').exists():
    saved=json.loads((CARD/'summary.json').read_text())
    objects,audit=load_verified_recorder_objects(CARD,expected_recorder_id=saved['recorder_id'],
        artifacts=['source/information_state.pkl','source/original_guidance_events.pkl','source/source_readback.pkl','source/source_manifest.pkl'])
    pd.testing.assert_frame_equal(objects['source/information_state.pkl'],state,check_exact=True)
    pd.testing.assert_frame_equal(objects['source/original_guidance_events.pkl'],available_events,check_exact=True)
    store=objects['source/source_manifest.pkl'];readback=objects['source/source_readback.pkl']
    digest,inventory=provider_content_hash(ROOT/plan['information_provider'])
    assert digest==saved['provider_content_sha256']
    rid=saved['recorder_id'];local=saved['recorder_local_dir'];uri=saved['recorder_uri']
else:
    store,readback=build_guidance_store(state,provider_uri=ROOT/plan['information_provider'],as_of=plan['as_of'],instrument=plan['information_instrument'],source=source)
    digest,inventory=provider_content_hash(ROOT/plan['information_provider'])
    uri=str(CARD/'qlib_recorder')
    with R.start(experiment_name=plan['experiment'],recorder_name='source_date_bound_qualification',uri=uri):
        rec=R.get_recorder()
        rec.log_params(fit_calls=0,native_cases=0,original_guidance_events=63,source_timing='conservative_date_bound',fact_manifest_sha256=source['fact_manifest_sha256'])
        rec.save_objects(artifact_path='source',**{'raw_facts.pkl':raw_facts,'normalized_events.pkl':normalized,'original_guidance_events.pkl':available_events,'information_state.pkl':state,'source_readback.pkl':readback,'source_manifest.pkl':store,'metadata_provenance.pkl':metadata_audit})
        rid=rec.id;local=rec.get_local_dir()
    assert rec.status=='FINISHED'
policy=dict(policy='slim_card_recorder_first',workflow_config='workflow.yaml',recorder_only_artifacts=['source/raw_facts.pkl','source/normalized_events.pkl','source/original_guidance_events.pkl','source/information_state.pkl','source/source_readback.pkl','source/source_manifest.pkl','source/metadata_provenance.pkl'])
summary=dict(experiment=plan['experiment'],status='conservative_date_bound_research_source_qualified',source_qualified_for_diagnostic_research=True,source_deployment_gate=False,promotion_decision='not_promoted',promotion_eligible=False,fit_calls=0,native_cases=0,original_guide_events=63,captured_filing_events=66,calendar_rows=len(state),known_rows=int(state.known.sum()),unknown_rows=int(state.known.eq(0).sum()),provider_content_sha256=digest,source=dict(provider_uri=plan['information_provider'],instrument=plan['information_instrument'],region='us',as_of=plan['as_of'],fact_manifest=plan['fact_manifest'],fact_manifest_sha256=source['fact_manifest_sha256']),recorder_id=rid,recorder_uri=uri,recorder_local_dir=local,recorder_status='FINISHED',artifact_policy=policy,limitations=['Retrospective primary-source textual extracts; original HTML byte identities not verified','Intraday first-public timestamps not verified; conservative publication-day lag explicitly used','Publisher total-growth assumptions include/exclude acquisitions as stated; not a comparable-perimeter fundamental growth forecast','Original guide only; all interim update coverage is not claimed','63 quarterly information observations are not3956 independent information events; no alpha or significance conclusion'])
(CARD/'summary.json').write_text(json.dumps(summary,indent=2,ensure_ascii=False)+'\n')
(CARD/'workflow.yaml').write_text(yaml.safe_dump(dict(experiment=plan['experiment'],source=summary['source'],recorder_uri=uri,recorder_id=rid,artifact_policy=policy),sort_keys=False))
(CARD/'selected_model.json').write_text(json.dumps(dict(model=None,status='source_qualification_only',promotion_decision='not_promoted'),indent=2)+'\n')
cols=['accession','release_date','reported_quarter','target_quarter','guidance_growth','guidance_scope','available_at']
available_events[cols].to_csv(CARD/'guidance_event_audit.csv',index=False)
entries=[]
for p in sorted((Path(local)/'artifacts').rglob('*')):
    if p.is_file() and not p.name.startswith('code_'):entries.append(dict(path=str(p.relative_to(CARD)),sha256=hashlib.sha256(p.read_bytes()).hexdigest(),bytes=p.stat().st_size))
(CARD/'verification_manifest.json').write_text(json.dumps(dict(recorder_id=rid,entries=entries),indent=2)+'\n')
conclusion=f"""# 原始季度指引来源：研究资格

已读66条披露事实，归一为63个连续目标财季的原始下一季指引。2014年5月6日提前收入与5月8日补发指引分开，未双计季度。原API/index时钟保留，按日期较晚者之后实际交易日09:31保守可用，未认证首次公布秒数。

Qlib源保留3956个价格信号日，{int(state.known.sum())}日已知、{int(state.known.eq(0).sum())}日未知，float32读回通过。只研究原始指导量，未覆盖/预测所有中期指导更新。并购包含/排除与公司口径保留，不造可比增长或未来实际。

研究来源达到声明的保守日期合同；0拟合、0回放，无收益风险或增量证据，不晋升。当前历史正文来自官方文本提取，未验证原HTML字节等价；首次盘中公开时点未认证，来源不获得生产签发。63个季度观测不能当作3956个独立信息事件。

下一步冻结单一信息增量及公平25/26特征对照；通过来源和原生DatasetH资格后，只跑一次固定预算，再以净超额和回撤门判定，不调参数救结果。

Recorder：{rid}；FINISHED。
"""
(CARD/'conclusion.md').write_text(conclusion)
print(json.dumps(summary,ensure_ascii=False,indent=2))


{
  "experiment": "nvda_original_guidance_qualification_20261011",
  "status": "conservative_date_bound_research_source_qualified",
  "source_qualified_for_diagnostic_research": true,
  "source_deployment_gate": false,
  "promotion_decision": "not_promoted",
  "promotion_eligible": false,
  "fit_calls": 0,
  "native_cases": 0,
  "original_guide_events": 63,
  "captured_filing_events": 66,
  "calendar_rows": 3956,
  "known_rows": 3924,
  "unknown_rows": 32,
  "provider_content_sha256": "40c37a958c02428eed9a378bfd7232b421956610462e07b37f7f5c610cb375a8",
  "source": {
    "provider_uri": "data/qlib/nvda_original_guidance_20261011_v1",
    "instrument": "NVDA_GUIDE_INFO",
    "region": "us",
    "as_of": "2026-09-25",
    "fact_manifest": "data/raw/external/nvda_guidance_facts_20261011_v1/manifest.json",
    "fact_manifest_sha256": "df09d3cf8fea92702f6f33a616b7e7c1fa2812ec6977289ee2afdc141f61b82a"
  },
  "recorder_id": "723e63dda85a424c9a0a74e1374dc480",
  "recorder_uri": "/Users/rziiiii/d

# 原始季度指引来源：研究资格

已读66条披露事实，归一为63个连续目标财季的原始下一季指引。2014年5月6日提前收入与5月8日补发指引分开，未双计季度。原API/index时钟保留，按日期较晚者之后实际交易日09:31保守可用，未认证首次公布秒数。

Qlib源保留3956个价格信号日，3924日已知、32日未知，float32读回通过。只研究原始指导量，未覆盖/预测所有中期指导更新。并购包含/排除与公司口径保留，不造可比增长或未来实际。

研究来源达到声明的保守日期合同；0拟合、0回放，无收益风险或增量证据，不晋升。当前历史正文来自官方文本提取，未验证原HTML字节等价；首次盘中公开时点未认证，来源不获得生产签发。63个季度观测不能当作3956个独立信息事件。

下一步冻结单一信息增量及公平25/26特征对照；通过来源和原生DatasetH资格后，只跑一次固定预算，再以净超额和回撤门判定，不调参数救结果。

Recorder：723e63dda85a424c9a0a74e1374dc480；FINISHED。
